[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_fuga.ipynb)

# Fuga de información, paso a paso

**Al terminar podrás** reconocer cuatro formas en que un dato del futuro se cuela a un modelo
(una columna, un promedio, un escalado y una partición al azar) y evitarlas con una sola regla y un `Pipeline`.

Esta hoja va junto con las slides de la sección de fuga: cada **Celda N** es el mismo código de la slide con ese número.
Córranla en orden, una celda a la vez, al mismo paso que la clase. Cada ajuste tarda unos segundos.
No se entrega ni se califica.

## Preparación

Corran las dos celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Celda 1. Los datos, partidos por fechas

Cada fila es una estación en una hora; queremos predecir `departures`, las bicis que salen en esa hora.
`est_hora_finde` es la interacción de la sección pasada: una etiqueta por estación, hora y tipo de día.
No imprime nada.

In [ ]:
# Celda 1
df = pd.read_parquet(RUTA_DATOS)
df["est_hora_finde"] = (df.station_id.astype(str) + "_"
                        + df.hour_of_day.astype(str) + "_" + df.is_weekend.astype(str))
train = df[df.particion == "train"]    # octubre a junio: para ajustar
julio = df[df.particion == "val"]      # julio: para medir y escoger
prueba = df[df.particion == "test"]    # agosto y septiembre: el examen final

## Celda 2. Dos funciones que usamos en toda la hoja

`ajusta` entrena la recta solo con las filas que le pasamos. `mae` dice por cuántas bicis falla, en promedio.
No imprime nada.

In [ ]:
# Celda 2
def ajusta(datos, columnas):           # la recta, ajustada SOLO con `datos`
    unos = ColumnTransformer([("oh", OneHotEncoder(), ["est_hora_finde"])],
                             remainder="passthrough")
    return make_pipeline(unos, LinearRegression()).fit(datos[columnas], datos.departures)

def mae(modelo, datos, columnas):      # el error promedio en bicis, con 2 decimales
    return round(mean_absolute_error(datos.departures, modelo.predict(datos[columnas])), 2)

## Celda 3. El modelo honesto

Solo columnas que ya se conocen a las 7:59: ayer a esta hora, hace una semana a esta hora, y la estación con su hora.
**Esperamos:** train 1.77, julio 1.85.

In [ ]:
# Celda 3
rezagos = ["departures_lag_24h", "departures_lag_168h"]   # ayer y hace una semana
honesto = rezagos + ["est_hora_finde"]                    # todo se conoce a las 7:59
modelo = ajusta(train, honesto)
print("train", mae(modelo, train, honesto), "  julio", mae(modelo, julio, honesto))

## Celda 4. Fuga 1, de columna: las llegadas de esa misma hora

`arrivals` son las bicis que llegan de 8 a 9. Viene en la misma fila, pero a las 7:59 nadie la conoce.
**Esperamos:** julio baja de 1.85 a 1.43.

In [ ]:
# Celda 4
con_llegadas = honesto + ["arrivals"]      # las bicis que LLEGAN de 8 a 9
modelo = ajusta(train, con_llegadas)
print("julio", mae(modelo, julio, con_llegadas))

## Celda 5. La misma fuga, completa

`net_flow` es salidas menos llegadas de esa hora. Con las llegadas y el flujo neto, la recta solo tiene que sumar.
**Esperamos:** julio 0.0.

In [ ]:
# Celda 5
con_neto = con_llegadas + ["net_flow"]     # salidas menos llegadas de 8 a 9
modelo = ajusta(train, con_neto)
print("julio", mae(modelo, julio, con_neto))

## Celda 6. ¿La prueba la detecta?

Medimos los dos modelos en agosto y septiembre, que nunca se usaron para nada.
**Esperamos:** honesto 1.85 y 1.93; con llegadas 1.43 y 1.53. La columna tramposa también gana en la prueba.

In [ ]:
# Celda 6
for nombre, cols in [("honesto", honesto), ("con llegadas", con_llegadas)]:
    modelo = ajusta(train, cols)
    print(nombre, "  julio", mae(modelo, julio, cols), "  prueba", mae(modelo, prueba, cols))

## Celda 7. Fuga 2, de cálculo: el promedio por grupo

La tabla de S1: el promedio de salidas de cada estación a cada hora y tipo de día. Una vez con solo train, otra con todo el año.
**Esperamos:** solo train 1.95; todo el año 1.91 (mejor, porque julio ya venía adentro del promedio).

In [ ]:
# Celda 7
grupo = ["station_id", "hour_of_day", "is_weekend"]
for nombre, datos in [("solo train", train), ("todo el año", df)]:
    tabla = datos.groupby(grupo).departures.mean().rename("prom")   # el promedio de cada grupo
    pred = julio.join(tabla, on=grupo).prom                         # se lo pegamos a julio
    print(nombre, round(mean_absolute_error(julio.departures, pred), 2))

## Celda 8. Fuga 3, al escalar

`StandardScaler` resta la media y divide entre la desviación. La media se aprende con datos: ¿con cuáles?
**Esperamos:** todo el año, media [4.07 4.05]; solo train, [3.96 3.97]. El MAE sale 2.13 en los dos.

In [ ]:
# Celda 8
for nombre, datos in [("todo el año", df), ("solo train", train)]:
    esc = StandardScaler().fit(datos[rezagos])        # aprende la media y la desviación
    modelo = Ridge().fit(esc.transform(train[rezagos]), train.departures)
    pred = modelo.predict(esc.transform(julio[rezagos]))
    print(nombre, esc.mean_.round(2), round(mean_absolute_error(julio.departures, pred), 2))

## Celda 9. Fuga 4, partir al azar una serie de tiempo

`train_test_split` revuelve las filas: horas de julio quedan para ajustar y horas de octubre para medir.
**Esperamos:** al azar 1.78; por fechas 1.85. El azar promete un error que en el futuro no se cumple.

In [ ]:
# Celda 9
hasta_julio = df[df.particion != "test"]                    # octubre a julio
train_azar, val_azar = train_test_split(hasta_julio, test_size=0.1, random_state=0)
print("al azar   ", mae(ajusta(train_azar, honesto), val_azar, honesto))
print("por fechas", mae(ajusta(train, honesto), julio, honesto))

## Celda 10. Cómo evitarla: un Pipeline

El escalador va dentro del modelo. Al llamar `fit` con train, todo lo que se aprende, se aprende con train.
**Esperamos:** media aprendida [3.96 3.97], la de train; julio 2.13.

In [ ]:
# Celda 10
modelo = make_pipeline(StandardScaler(), Ridge())   # el escalador va DENTRO del modelo
modelo.fit(train[rezagos], train.departures)        # fit solo ve train
print("media aprendida", modelo[0].mean_.round(2))
print("julio", mae(modelo, julio, rezagos))

## La lista de revisión

Antes de creerle a un modelo, cuatro preguntas:

1. ¿A qué hora conozco cada columna? Si se conoce después del momento de predecir, fuera.
2. ¿Con qué filas calculé cada promedio, tabla o escala? Solo con train.
3. ¿Partí por fechas? En series de tiempo, el pasado ajusta y el futuro mide.
4. ¿El error bajó de golpe? Si sale demasiado bien, quiten columnas una por una hasta encontrar la que hace el milagro.

## Tres preguntas

Escriban la letra en la hoja, debajo de cada pregunta (doble clic para escribir).

**1. A las 7:59 queremos predecir cuántas bicis salen de 8 a 9. ¿Cuál columna tiene fuga?**

- a) departures_lag_24h: las salidas de ayer de 8 a 9
- b) arrivals: las llegadas de hoy de 8 a 9
- c) hour_of_day: la hora del día
- d) is_weekend: si es sábado o domingo

**Su respuesta (doble clic para escribir):** ___

**2. Un modelo da 1.43 en julio y 1.53 en la prueba. El mejor honesto da 1.85 y 1.93. ¿Qué concluyen?**

- a) Que es mejor: ganó también en la prueba
- b) Que la prueba está mal hecha
- c) Que hay que revisar a qué hora se conoce cada columna: la prueba no atrapa una columna tramposa
- d) Que hay que medirlo en train

**Su respuesta (doble clic para escribir):** ___

**3. ¿Con qué filas se hace el fit del StandardScaler?**

- a) Con todo el año, para que la media sea más exacta
- b) Solo con train, y lo más seguro es ponerlo dentro de un Pipeline
- c) Con julio, que es donde medimos
- d) Da igual: el MAE salió igual en la Celda 8

**Su respuesta (doble clic para escribir):** ___